In [1]:
# All paths in this notebook are relative to the repository root; anchor the working directory there
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Corridor Profile of Potential Movements — Survey-Only 2022 Matrices (Total, Transit, Taxi-type)

Directional link profiles along the LRT corridor for the **2022 layers** of `THS_2017_three_mode_2022.ipynb`, built the same way as the profile of the leveled hybrid-based matrix (`Output/transit/corridor_link_flows_2022.csv`, commit 7d07a91):

- the corridor areas (`IsLRT_Corridor = 1` in `Input/Submatrix_tazs.xlsx`) in `AggAreaCode` order form the line sequence — the same 18 areas as before (Adi, Alon Hagalil and Tzipori, which the earlier 25-area matrix had filtered out as noise, are left out for comparability; review §11 asks for them back with flagged uncertainty once the line sequence is settled);
- every OD pair with **both ends on the sequence** is assigned to every link between its origin and destination, in the direction of travel — direction 1 → 23 is Tirat Carmel → Nazareth, 23 → 1 the reverse;
- three profiles: **total** = car + bus + taxi-type + rail, **transit** = bus (calibrated) + rail (survey, door-to-door), and **taxi-type** on its own.

**What these numbers are.** Each value is the number of 06:00–09:00 trips between line areas that *could* traverse a link if every such trip used the line — a three-hour total of potential movements. It is not a passenger load: it says nothing about station access, route choice against parallel services, peaking within the three hours, or trips with one end off the line (review §15).

The earlier hybrid-based profiles are drawn as dashed lines for comparison. Their transit layer was the RavKav × OnBoard bus matrix plus the station-based train matrix, so differences in the transit profile reflect both the source (survey vs ticketing frame) and the calibration.

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, ORANGE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
IN = 'Output/ths2017/three_mode_2022'; OUT = IN
os.makedirs('Output/figures', exist_ok=True)

def load_area(name):
    m = pd.read_csv(f'{IN}/{name}_2022_area.csv', index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int); return m
car, bus, taxi, rail = load_area('car'), load_area('bus'), load_area('taxi'), load_area('rail')
total, transit = car + bus + taxi + rail, bus + rail        # transit = scheduled bus (RavKav-ticketed) + rail; taxi-type is a separate layer (review §9)
legend = pd.read_csv('Output/ths2017/study_taz/study_taz_area_legend.csv' if os.path.exists('Output/ths2017/study_taz/study_taz_area_legend.csv') else 'Output/ths2017/study_taz/submatrices/area_legend.csv').set_index('AggAreaCode')
ref_total = pd.read_csv('Output/transit/corridor_link_flows_2022.csv')
ref_transit = pd.read_csv('Output/transit/corridor_link_flows_transit_2022.csv')
SEQ = list(ref_total['from_code']) + [int(ref_total['to_code'].iloc[-1])]          # the same 18-area line sequence as the earlier profile
assert all(legend.loc[a, 'IsLRT_Corridor'] == 1 for a in SEQ)
names = legend['AggAreaName']
print("line sequence:", ' → '.join(f'{a} {names[a]}' for a in SEQ))
print(f"2022 area matrices: total {total.values.sum():,.0f} trips in the 28 sub-areas (car {car.values.sum():,.0f}, bus {bus.values.sum():,.0f}, taxi-type {taxi.values.sum():,.0f}, rail {rail.values.sum():,.0f})")

line sequence: 1 TiratCarmel → 2 Matam → 3 Neot Peres → 4 Neve David → 5 Ein Hayam → 6 Bat Galim → 7 Kiryat Eliezer → 8 Hamoshava → 9 Lower City → 10 Hadar Carmel → 11 Namal → 12 Neve Yosef → 13 Hamifrats → 14 Hutzot → 16 Kiryat Ata Center → 17 Kiryat Bialik South → 19 Shefaram → 23 Nazareth Area
2022 area matrices: total 209,929 trips in the 28 sub-areas (car 182,374, bus 26,775, taxi-type 676, rail 104)


## Link flows

In [3]:
def link_flows(m, seq):
    pos = {a: i for i, a in enumerate(seq)}
    sub = m.reindex(index=seq, columns=seq).fillna(0).values
    n = len(seq); f_fwd, f_bwd = np.zeros(n - 1), np.zeros(n - 1)
    for i in range(n):
        for j in range(n):
            if i < j: f_fwd[i:j] += sub[i, j]          # links i..j-1 in direction 1 -> 23
            elif i > j: f_bwd[j:i] += sub[i, j]        # links j..i-1 in direction 23 -> 1
    out = pd.DataFrame({'from_code': seq[:-1], 'to_code': seq[1:], 'from_area': [names[a] for a in seq[:-1]], 'to_area': [names[a] for a in seq[1:]],
                        'flow_dir_1_to_23': f_fwd.round(0), 'flow_dir_23_to_1': f_bwd.round(0)})
    out['total_both_dir'] = out['flow_dir_1_to_23'] + out['flow_dir_23_to_1']
    internal = sub[~np.eye(n, dtype=bool)].sum() + np.trace(sub)
    return out, internal

prof_total, int_total = link_flows(total, SEQ)
prof_transit, int_transit = link_flows(transit, SEQ)
prof_taxi, int_taxi = link_flows(taxi, SEQ)
prof_taxi.to_csv(f'{OUT}/corridor_link_flows_taxi_2022.csv', index=False)
prof_total.to_csv(f'{OUT}/corridor_link_flows_total_2022.csv', index=False)
prof_transit.to_csv(f'{OUT}/corridor_link_flows_transit_2022.csv', index=False)
for name, p, internal, ref, tot in [('total (car + bus + taxi + rail)', prof_total, int_total, ref_total, total), ('transit (bus + rail)', prof_transit, int_transit, ref_transit, transit), ('taxi-type (separate layer)', prof_taxi, int_taxi, ref_transit, taxi)]:
    k = p['total_both_dir'].idxmax()
    print(f"{name}: corridor-internal demand {internal:,.0f} of {tot.values.sum():,.0f} sub-area trips; peak link {p.loc[k, 'from_area']} – {p.loc[k, 'to_area']} "
          f"({p.loc[k, 'flow_dir_1_to_23']:,.0f} / {p.loc[k, 'flow_dir_23_to_1']:,.0f}); earlier hybrid-based peak {ref['total_both_dir'].max():,.0f} both directions")
cmp = prof_total[['from_area', 'to_area', 'flow_dir_1_to_23', 'flow_dir_23_to_1']].copy()
cmp.columns = ['from', 'to', 'total 1→23', 'total 23→1']
cmp['hybrid-based 1→23'] = ref_total['flow_dir_1_to_23']; cmp['hybrid-based 23→1'] = ref_total['flow_dir_23_to_1']
cmp['transit 1→23'] = prof_transit['flow_dir_1_to_23']; cmp['transit 23→1'] = prof_transit['flow_dir_23_to_1']
cmp['taxi-type 1→23'] = prof_taxi['flow_dir_1_to_23']; cmp['taxi-type 23→1'] = prof_taxi['flow_dir_23_to_1']
cmp['ticketing-based transit 1→23'] = ref_transit['flow_dir_1_to_23']; cmp['ticketing-based transit 23→1'] = ref_transit['flow_dir_23_to_1']
cmp['transit share 1→23'] = cmp['transit 1→23'] / cmp['total 1→23']; cmp['transit share 23→1'] = cmp['transit 23→1'] / cmp['total 23→1']
cmp.to_csv(f'{OUT}/corridor_link_flows_comparison_2022.csv', index=False, float_format='%.3f')
cmp.round(2)

total (car + bus + taxi + rail): corridor-internal demand 68,662 of 209,929 sub-area trips; peak link Bat Galim – Kiryat Eliezer (5,925 / 6,330); earlier hybrid-based peak 15,247 both directions
transit (bus + rail): corridor-internal demand 9,463 of 26,879 sub-area trips; peak link Ein Hayam – Bat Galim (1,842 / 1,162); earlier hybrid-based peak 4,240 both directions
taxi-type (separate layer): corridor-internal demand 313 of 676 sub-area trips; peak link Hamoshava – Lower City (0 / 178); earlier hybrid-based peak 4,240 both directions


,from,to,total 1→23,total 23→1,hybrid-based 1→23,hybrid-based 23→1,transit 1→23,transit 23→1,taxi-type 1→23,taxi-type 23→1,ticketing-based transit 1→23,ticketing-based transit 23→1,transit share 1→23,transit share 23→1
0,TiratCarmel,Matam,3768.0,1384.0,5085.0,1075.0,915.0,405.0,0.0,0.0,776.0,192.0,0.24,0.29
1,Matam,Neot Peres,2973.0,3376.0,4136.0,2358.0,930.0,1184.0,0.0,0.0,1047.0,419.0,0.31,0.35
2,Neot Peres,Neve David,2968.0,3880.0,3924.0,3194.0,916.0,1488.0,0.0,96.0,1037.0,874.0,0.31,0.38
3,Neve David,Ein Hayam,4345.0,3784.0,6152.0,4746.0,1422.0,1369.0,0.0,40.0,1632.0,1629.0,0.33,0.36
4,Ein Hayam,Bat Galim,6092.0,3201.0,6761.0,3913.0,1842.0,1162.0,0.0,0.0,1853.0,1648.0,0.30,0.36
5,Bat Galim,Kiryat Eliezer,5925.0,6330.0,7671.0,7576.0,1605.0,1373.0,0.0,126.0,1656.0,2584.0,0.27,0.22
6,Kiryat Eliezer,Hamoshava,6160.0,4727.0,7103.0,6025.0,1601.0,1101.0,0.0,132.0,1554.0,2398.0,0.26,0.23
7,Hamoshava,Lower City,5097.0,3458.0,5911.0,4629.0,1647.0,1117.0,0.0,178.0,1461.0,2326.0,0.32,0.32
8,Lower City,Hadar Carmel,3974.0,4171.0,5635.0,6060.0,1176.0,1309.0,0.0,178.0,1175.0,2736.0,0.30,0.31
9,Hadar Carmel,Namal,2691.0,3535.0,3856.0,4870.0,866.0,1243.0,0.0,123.0,725.0,2652.0,0.32,0.35


In [4]:
def profile_chart(p, ref, title, ylabel, fname, note, ref_label='earlier hybrid-based matrix', extra=None):
    n = len(p) + 1; x = np.arange(n)
    fig, ax = plt.subplots(figsize=(15, 7.2), facecolor='white')
    for col, color, lab in [('flow_dir_1_to_23', BLUE, f'Direction 1 → 23  ({names[SEQ[0]]} → {names[SEQ[-1]]})'), ('flow_dir_23_to_1', ORANGE, f'Direction 23 → 1  ({names[SEQ[-1]]} → {names[SEQ[0]]})')]:
        y = p[col].values
        ax.stairs(y, x, color=color, linewidth=2, label=lab, baseline=0, fill=True, alpha=0.18)
        ax.stairs(y, x, color=color, linewidth=2)
        ax.stairs(ref[col].values, x, color=color, linewidth=1.3, linestyle='--', alpha=0.85, label=lab.split('  ')[0] + ' — ' + ref_label)
        k = int(np.argmax(y)); ax.text(k + 0.5, y[k] * 1.02, f'{y[k]:,.0f}', ha='center', va='bottom', color=color, fontsize=10, fontweight='bold')
        if extra is not None: ax.stairs(extra[0][col].values, x, color=color, linewidth=1.2, linestyle=':', label=lab.split('  ')[0] + ' — ' + extra[1])
    ax.set_xticks(x); ax.set_xticklabels([f'{a} · {names[a]}' for a in SEQ], rotation=55, ha='right', fontsize=9, color=INK2)
    ax.set_xlim(0, n - 1); ax.set_ylim(0, max(p[['flow_dir_1_to_23', 'flow_dir_23_to_1']].values.max(), ref[['flow_dir_1_to_23', 'flow_dir_23_to_1']].values.max()) * 1.12)
    ax.grid(True, color=GRID, linewidth=0.6, axis='y'); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2); ax.set_ylabel(ylabel, color=INK2); ax.set_title(title, color=INK, fontsize=14)
    ax.legend(frameon=False, fontsize=10, loc='upper right')
    fig.text(0.5, -0.02, note, ha='center', color=INK2, fontsize=9)
    plt.tight_layout(); fig.savefig(f'Output/figures/{fname}', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()

profile_chart(prof_total, ref_total, 'LRT corridor — potential movements along the line, 06:00–09:00 (3-hour totals) — survey-only matrix, 2022 base, all modes',
              'trips crossing the link in 3 hours (car + bus + taxi + rail)', 'corridor_flow_profile_total_survey2022.png',
              f'Corridor areas in AggAreaCode order; each OD pair with both ends on the line is assigned to every link between them. Corridor-internal demand {int_total:,.0f} trips. '
              'Dashed: the earlier profile of the leveled hybrid-based all-mode matrix. These are potential movements between line areas, not passenger loads.')
profile_chart(prof_transit, ref_transit, 'LRT corridor — transit potential movements along the line, 06:00–09:00 (3-hour totals) — survey-only matrix, 2022 base, bus + rail',
              'transit trips crossing the link in 3 hours (bus + rail; taxi-type dotted)', 'corridor_flow_profile_transit_survey2022.png',
              f'Corridor-internal transit demand {int_transit:,.0f} trips (taxi-type, shown dotted, a further {int_taxi:,.0f}). Bus = survey bus calibrated to RavKav × OnBoard (segmented coverage rule); rail = survey door-to-door. '
              'Dashed: the earlier ticketing-based transit profile (RavKav × OnBoard bus + station-based train). Potential movements, not passenger loads.', ref_label='earlier ticketing-based transit', extra=(prof_taxi, 'taxi-type (separate layer)'))

## Notes

- **Same construction, different matrices.** The earlier profile used the leveled hybrid-based all-mode matrix (survey car / other Furnessed to 2022 + RavKav × OnBoard bus + train). This one uses the survey-only car / bus / taxi / rail set: car from the trips file grown to 2022, bus calibrated to ticketing segment by segment where ticketing coverage is credible, rail from the survey. Differences in the total profile come mostly from the car layer's source (population / employment TAZ split, no cellular structure) and from the "other" modes, which the earlier matrix included and this one does not.
- **Taxi-type is not in the transit profile.** The 9,316 sub-area taxi-type trips (3,606 corridor-internal; up to 1,415 on the Ein Hayam – Bat Galim link towards Nazareth) are the dotted lines. Whether any of them belong in an LRT market depends on what modes 5 / 8 are — to be settled with the codebook, not by summation.
- **Transit frame.** The survey bus layer starts trips at the doorstep; the ticketing matrix starts them at the boarding stop, so hub areas (Hamifrats, Neve Yosef) draw more flow in the ticketing-based profile than here.
- **Corridor-internal only, three-hour totals.** Trips with one end off the line are not on the profile; the extended market of `NoBuild_and_LRT_market.ipynb` covers them. Peak-hour loads need within-period peaking and a service-response model.